In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(columns=['id','Unnamed: 32'], inplace = True)

In [5]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [11]:
x = df.iloc[:,1:]
y = df.iloc[:,0]

In [14]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [15]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [16]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

## Numpy array to pytorch tensor

In [17]:
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [19]:
x_train_tensor.shape

torch.Size([455, 30])

In [21]:
y_train_tensor.shape

torch.Size([455])

## Define the model

In [27]:
class SimpleNN():
    def __init__(self, x):

        self.weights = torch.rand(x.shape[1],1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.rand(1, dtype=torch.float64, requires_grad= True)

    def forward_pass(self,x):
        z = torch.matmul(x, self.weights)+self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    

    def loss(self, y_pred, y):
        # clamp predictions to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)

        #calculate loss
        loss = -(y*torch.log(y_pred) + (1 - y)*torch.log(1 - y_pred)).mean()
        return loss



    

## Important parameters

In [34]:
learning_rate = 0.1
epochs = 25

In [35]:
# create model
model = SimpleNN(x_train_tensor)

#define loops
for epoch in range(epochs):
    # forward pass
    y_pred = model.forward_pass(x_train_tensor)
    # print(y_pred)

    #loss calculation
    loss = model.loss(y_pred, y_test_tensor)
    # print(loss)

    # backward pass
    loss.backward()

    # parameters update
    with torch.no_grad():
        model.weights -= learning_rate*model.weights.grad
        model.bias -= learning_rate*model.bias.grad

    # zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # print loss in each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')
    




Epoch: 1, Loss: 3.641299076428562
Epoch: 2, Loss: 3.525510691251497
Epoch: 3, Loss: 3.4090680705751497
Epoch: 4, Loss: 3.2888505111666753
Epoch: 5, Loss: 3.165644445112995
Epoch: 6, Loss: 3.042993755198348
Epoch: 7, Loss: 2.913192576853326
Epoch: 8, Loss: 2.779078794500927
Epoch: 9, Loss: 2.640968920686214
Epoch: 10, Loss: 2.495590486599934
Epoch: 11, Loss: 2.3483424399703883
Epoch: 12, Loss: 2.206000877860018
Epoch: 13, Loss: 2.063293075853649
Epoch: 14, Loss: 1.926746615032654
Epoch: 15, Loss: 1.797270068594469
Epoch: 16, Loss: 1.6758669029493685
Epoch: 17, Loss: 1.563350984347853
Epoch: 18, Loss: 1.4632094518004324
Epoch: 19, Loss: 1.3748994091414246
Epoch: 20, Loss: 1.2963643656527384
Epoch: 21, Loss: 1.229482357348486
Epoch: 22, Loss: 1.1712112141617963
Epoch: 23, Loss: 1.1219769933441284
Epoch: 24, Loss: 1.0799576712224839
Epoch: 25, Loss: 1.0450780579749201


In [36]:
model.weights

tensor([[ 0.1394],
        [-0.1033],
        [ 0.5316],
        [-0.4200],
        [ 0.0146],
        [ 0.2394],
        [-0.3621],
        [-0.0177],
        [ 0.1071],
        [-0.0315],
        [ 0.5099],
        [ 0.0535],
        [ 0.1329],
        [ 0.2418],
        [ 0.6024],
        [-0.0503],
        [ 0.4155],
        [-0.3645],
        [ 0.6833],
        [ 0.3610],
        [-0.3757],
        [ 0.2998],
        [-0.0451],
        [ 0.4041],
        [ 0.0791],
        [-0.4281],
        [ 0.4112],
        [-0.1581],
        [-0.1972],
        [-0.2694]], dtype=torch.float64, requires_grad=True)

In [37]:
model.bias

tensor([0.5523], dtype=torch.float64, requires_grad=True)

## Evaluation

In [38]:
# model evaluation

with torch.no_grad():
    y_pred = model.forward_pass(x_test_tensor)
    y_pred = (y_pred > 0.9).float()

    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.5861803889274597
